# Buổi 3.8: Embeddings, pooling, attention và Transformers
Padding hoặc token tương lai có được tham gia attention không?

## Kiến thức
- Embeddings biến IDs thành vectors; pooling phải bỏ padding.
- Q/K xác định weights; V là nội dung được tổng hợp; softmax theo keys.
- Transformer thêm multi-head, FFN, residual, normalization và thông tin vị trí.

**Công thức:** Attention(Q,K,V)=softmax(QKᵀ/sqrt(dk)+mask)V


In [1]:
# -*- coding: utf-8 -*-
import torch,math
from torch import nn
torch.manual_seed(42)
torch.set_num_threads(1)

# Hai tokens, dimension2. Giá trị bên phải là token sẽ bị mask.
Q = torch.tensor([[1.,0.]])
K = torch.tensor([[1.,0.],[0.,1.]])
V = torch.tensor([[10.,0.],[0.,20.]])
score = Q@K.T/math.sqrt(2)
weights = torch.softmax(score,dim=-1)
print("Weights:",weights.round(decimals=4).tolist())
print("Output:",(weights@V).round(decimals=4).tolist())
masked = score.masked_fill(torch.tensor([[False,True]]),float("-inf"))
print("Masked output:",(torch.softmax(masked,-1)@V).tolist())
layer = nn.TransformerEncoderLayer(d_model=4,nhead=2,dim_feedforward=8,
                                   dropout=0,batch_first=True)
layer.eval(); tokens = torch.arange(12,dtype=torch.float32).reshape(1,3,4)
with torch.no_grad(): print("Transformer shape:",tuple(layer(tokens).shape))
assert (torch.softmax(masked,-1)@V).tolist()==[[10.,0.]]


Weights: [[0.6697999835014343, 0.3301999866962433]]
Output: [[6.6975998878479, 6.604800224304199]]
Masked output: [[10.0, 0.0]]
Transformer shape: (1, 3, 4)


## Đọc kết quả
Code/shape và các câu hỏi theo sách; chưa phải tự train LM lớn.
## Nguồn
KHTN chương 19–20; AIO 37.1–37.3.
AIO 37.1 các phần thực hành Transformer;37.4 trắc nghiệm, gồm text/image Transformer.


## Hai bài trên lớp
1. Mean token vectors chỉ tại mask=1; nếu toàn 0 trả zero vector.
Input: `{"tokens": [[1, 0], [0, 1], [9, 9]], "mask": [1, 1, 0]}`
2. Scaled-dot attention QK^T/sqrt(dk), softmax theo keys; allow=false bịmask. Mỗi query có ít nhất 1 key được phép.
Input: `{"q": [[1, 0]], "k": [[1, 0], [0, 1]], "v": [[10, 0], [0, 20]], "allow": [[true, false]]}`

Viết cell mới để làm. Đối chiếu trong BAI_HOC.html sau khi thử.
## BTVN
- BTVN 1: BTVN_01.txt
- BTVN 2: BTVN_02.txt
- BTVN 3: BTVN_03.txt
